Biometrich Technologies and Behavioural Security
# **<center>Tutorial 3 - Face Recognition</center>**
## **<center>Part 1: The textural descriptor BSIF</center>**






## Step 1: Load data and packages in your VM

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_lfw_people
from sklearn.metrics import classification_report
import numpy as np
import cv2
from scipy import signal
import scipy.io
import math
from IPython.display import clear_output


In [ ]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
#BSIF Filters download code
downloaded = drive.CreateFile({'id': '1CPk07Oqn5gmKn8yXaEns-0yZc5ZAs3Su'})   # replace the id with id of file you want to access
downloaded.GetContentFile('texturefilters.zip')        # replace the file name with your file
!unzip texturefilters
print('Done! Press the Refresh button if files are not visible.')
clear_output()

## Step 2: BSIF method definition

In [ ]:
def bsif(img, filterpath):
    f = scipy.io.loadmat(filterpath)
    texturefilters = f.get('ICAtextureFilters')

    # Initialize
    img = img.astype("float");
    numScl = np.shape(texturefilters)[2]
    codeImg = np.ones(np.shape(img))

    # Make spatial coordinates for sliding window
    r = int(math.floor(np.shape(texturefilters)[0] / 2))

    # Wrap image (increase image size according to maximum filter radius by wrapping around)
    upimg = img[0:r, :]
    btimg = img[-r:, :]
    lfimg = img[:, 0:r]
    rtimg = img[:, -r:]
    cr11 = img[0:r, 0:r]
    cr12 = img[0:r, -r:]
    cr21 = img[-r:, 0:r]
    cr22 = img[-r:, -r:]
    imgWrap = np.vstack(
        (np.hstack((cr22, btimg, cr21)), np.hstack((rtimg, img, lfimg)), np.hstack((cr12, upimg, cr11))))

    # Loop over scales
    for i in range(numScl):
        tmp = texturefilters[:, :, numScl - i - 1]
        ci = signal.convolve2d(imgWrap, np.rot90(tmp, 2), mode='valid')
        t = np.multiply(np.double(ci > 0), 2 ** i)
        codeImg = codeImg + t

    hist_bsif = np.histogram(codeImg.ravel(), bins=np.arange(1,(2**numScl)+2))
    hist_bsif = hist_bsif[0]
    # normalize the histogram
    hist_bsif = hist_bsif/(hist_bsif.sum() + 1e-7)


    return codeImg, hist_bsif

Summing up, there are two main parameters in the BSIF descriptor:

1.   Filter window size **l**
2.   Length **n** of the bit string


In this tutorial we choose the window size equal to 7x7, and features number 4096 (corresponding to 12 bits)

In [ ]:
file_path = "/content/texturefilters/ICAtextureFilters_7x7_12bit.mat"

## Step 3: Feature Extraction

In [ ]:
#Initialize data and labels array
lfw_dataset = fetch_lfw_people(min_faces_per_person=100)

_, h, w = lfw_dataset.images.shape
X = lfw_dataset.data
labels = lfw_dataset.target
target_names = lfw_dataset.target_names


In [ ]:
data = []
for image in X:
  image=image.reshape((h, w))

  img_bsif,hist = bsif(image,file_path)

  data.append(hist)
print(np.shape(data))

(1140, 4096)


## Step 4: Classification with SVM

In [ ]:
#Split data array in train and test data for SVM

from sklearn.model_selection import train_test_split
data = np.array(data)
labels = np.array(labels)
X_train,X_test,y_train,y_test = train_test_split(data,labels,test_size = 0.2)

In [ ]:
#Train the SVM classifier

from sklearn import svm
clf = svm.SVC()
clf.fit(X_train, y_train)

SVC()

In [ ]:
#Predict test labels and accuracy

y_pred = clf.predict(X_test)
from sklearn.metrics import accuracy_score
acc='Total Accuracy: %.2f %%' % (accuracy_score(y_test, y_pred)*100)
print(acc)

Total Accuracy: 85.09 %
